# CNN Chest X-Ray Classifier with MLflow Model Versioning
Trains the same CNN architecture with **Adam** and **SGD** optimizers, logs every run to MLflow, registers each model version, and automatically aliases the best one as `@champion`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import mlflow
import mlflow.tensorflow
from mlflow import MlflowClient
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    f1_score, accuracy_score, precision_score, recall_score
)

## Configuration
Edit the paths and hyper-parameters here before running.

In [ ]:
TRAIN_DIR   = '/home/arshia/Downloads/archive/train'
VAL_DIR     = '/home/arshia/Downloads/archive/val'
TEST_DIR    = '/home/arshia/Downloads/archive/test'
EPOCHS      = 4
BATCH_SIZE  = 32
IMG_SIZE    = (224, 224)
MODEL_NAME  = "chest-xray-cnn"   # MLflow Registered Model name

# ── Optimizers to compare ──────────────────────────────────────────────────
# Add / remove entries here to run more experiments
OPTIMIZER_CONFIGS = [
    {
        "name": "adam",
        "optimizer": tf.keras.optimizers.Adam(learning_rate=1e-3),
        "description": "Adam optimizer, lr=1e-3"
    },
    {
        "name": "sgd",
        "optimizer": tf.keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9),
        "description": "SGD optimizer, lr=1e-2, momentum=0.9"
    },
]

## Data 

In [ ]:
def build_generators():
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=10,
        zoom_range=0.1,
        horizontal_flip=True
    )
    training_set = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='binary'
    )

    valid_datagen = ImageDataGenerator(rescale=1./255)
    validation_set = valid_datagen.flow_from_directory(
        VAL_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='binary'
    )

    test_datagen = ImageDataGenerator(rescale=1./255)
    test_set = test_datagen.flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
    )

    return training_set, validation_set, test_set

## Model

In [ ]:
def build_model(optimizer):
    model = tf.keras.models.Sequential([
        # Layer 1
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
        tf.keras.layers.MaxPooling2D(2, 2),

        # Layer 2
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),

        # Layer 3
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'), # Increased units for deeper nets
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

## Accuracy

In [ ]:
def evaluate_model(model, test_set):
    y_prob = model.predict(test_set).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = test_set.classes

    metrics = {
        "test_accuracy":  accuracy_score(y_true, y_pred),
        "test_precision": precision_score(y_true, y_pred, zero_division=0),
        "test_recall":    recall_score(y_true, y_pred, zero_division=0),
        "test_f1":        f1_score(y_true, y_pred, zero_division=0),
        "test_roc_auc":   roc_auc_score(y_true, y_prob),
    }

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred,
                                target_names=test_set.class_indices.keys()))

    return metrics, y_true, y_prob, cm

## MLflow Experiment Loop
Runs one MLflow **Run** per optimizer, registers the model, and tags each version.

In [ ]:
mlflow.set_experiment("chest-xray-cnn-comparison")
client = MlflowClient()

# Create the registered model (idempotent)
try:
    client.create_registered_model(
        MODEL_NAME,
        description="Binary chest X-ray classifier (pneumonia vs normal)"
    )
    print(f"Created registered model: {MODEL_NAME}")
except mlflow.exceptions.MlflowException:
    print(f"Registered model '{MODEL_NAME}' already exists – reusing it.")

training_set, validation_set, test_set = build_generators()
run_ids = {}

for cfg in OPTIMIZER_CONFIGS:
    opt_name = cfg["name"]
    print(f"\n{'='*55}")
    print(f"  Training with optimizer: {opt_name.upper()}")
    print(f"{'='*55}")

    with mlflow.start_run(run_name=f"cnn_{opt_name}") as run:
        run_id = run.info.run_id
        run_ids[opt_name] = run_id

        # Log hyper-parameters
        mlflow.log_params({
            "optimizer":    opt_name,
            "epochs":       EPOCHS,
            "batch_size":   BATCH_SIZE,
            "image_size":   f"{IMG_SIZE[0]}x{IMG_SIZE[1]}",
            "dropout":      0.5,
            "conv_filters": 32,
            "dense_units":  128,
        })
        mlflow.set_tag("description", cfg["description"])

        # Build model
        model = build_model(cfg["optimizer"])

        # Keras callback to log per-epoch metrics
        class MLflowEpochLogger(tf.keras.callbacks.Callback):
            def on_epoch_end(self, epoch, logs=None):
                logs = logs or {}
                mlflow.log_metrics({
                    "epoch_train_loss":     logs.get("loss", 0),
                    "epoch_train_accuracy": logs.get("accuracy", 0),
                    "epoch_val_loss":       logs.get("val_loss", 0),
                    "epoch_val_accuracy":   logs.get("val_accuracy", 0),
                }, step=epoch)

        history = model.fit(
            training_set,
            validation_data=validation_set,
            epochs=EPOCHS,
            callbacks=[MLflowEpochLogger()]
        )

        # Evaluate
        results = model.evaluate(test_set, return_dict=True)
        mlflow.log_metric("test_loss", results["loss"])

        metrics, y_true, y_prob, cm = evaluate_model(model, test_set)
        mlflow.log_metrics(metrics)
    
        
        # Register model version
        mlflow.tensorflow.log_model(
            model,
            artifact_path="model",
            registered_model_name=MODEL_NAME,
        )

        # Tag the new version
        versions = client.search_model_versions(
            f"name='{MODEL_NAME}' and run_id='{run_id}'"
        )
        if versions:
            mv = versions[0]
            client.set_model_version_tag(MODEL_NAME, mv.version, "optimizer", opt_name)
            client.update_model_version(
                MODEL_NAME, mv.version,
                description=(
                    f"{cfg['description']} | "
                    f"F1={metrics['test_f1']:.4f} | "
                    f"AUC={metrics['test_roc_auc']:.4f}"
                )
            )
            print(f"\n✓ Registered as '{MODEL_NAME}' version {mv.version}")

## Compare Versions & Promote Best Model

In [ ]:
print("\n" + "="*55)
print("  Comparing runs and promoting best model")
print("="*55)

best_f1      = -1
best_version = None

for version in client.search_model_versions(f"name='{MODEL_NAME}'"):
    run  = client.get_run(version.run_id)
    f1   = run.data.metrics.get("test_f1", 0)
    print(f"  Version {version.version} | "
          f"optimizer={version.tags.get('optimizer','?')} | "
          f"F1={f1:.4f}")
    if f1 > best_f1:
        best_f1      = f1
        best_version = version.version

if best_version:
    client.set_registered_model_alias(MODEL_NAME, "champion", best_version)
    print(f"\n  Version {best_version} aliased as 'champion' (F1={best_f1:.4f})")

In [ ]:
model.save("pneumonia_model.keras")